# $\sigma_{PS}$ condensate vs mass, all Nf (gsq=8, nt128 L4)

Reads `h0/condensate/etadag_xi` from the massive Real-$m$ disc data
(`data_Nf{nf}_..._mRe{m}..._vmRe{m}.../corr_ylm_disc_tb2_nhits1/`).
Convention = per-mass notebook's $\sigma_{PS}$ cell: $\mathrm{dens}=\mathrm{etadag\_xi}/(N_t\cdot 4\pi)$,
$\sigma_{PS}=\mathrm{dens}+\overline{\mathrm{dens}}$; **contact-subtracted $+2$**
(`condensate_contact_massive_claude.md` Sec 10, mass-independent) = SSB order parameter.
Config-jackknife (1 hit/config).  **x-axis = effective $m_F$**.  Light Family-B rescaled so
effective $m_F=0.2,0.1,0.05,0.01$ at every L (dirs mRe0.211450..; Nf=2,4,6) -- NOTE L4 light
condensate may not be measured yet (defensive NaN).  **Heavy** $m_F=0.1095,0.2191,0.3286$
(Nf=2 only, red diamond; effective $m_F$ at L4).

In [1]:
import h5py, numpy as np, glob, math
import matplotlib.pyplot as plt

Nt = 128
sumA = 4.0*math.pi
norm = Nt*sumA
# x-axis = EFFECTIVE m_F.  Light Family-B rescaled so effective m_F = 0.2/0.1/0.05/0.01 at every L
# (dirs mRe0.211450..).  L4 light disc/condensate may be absent -> NaN (defensive).
MASS_DIR = ['0.211450', '0.105725', '0.052862', '0.010572']
MASS = np.array([0.2, 0.1, 0.05, 0.01])
NFS = [2, 4, 6]
NFMARK = ['o', 's', '^']

def esn(nf, m): return f'data_Nf{nf}_gsq8.000000at0.200000nu01.000000mRe{m}mIm0.000000nt128L4_vmRe{m}vmIm0.000000/'
def disc_files(nf, m): return sorted(glob.glob(esn(nf, m)+'corr_ylm_disc_tb2_nhits1/corr.*.h0.h5'))

def cond_cfg(files):
    out = []
    for fn in files:
        with h5py.File(fn, 'r') as f:
            out.append(f['h0/condensate/etadag_xi/real'][0] + 1j*f['h0/condensate/etadag_xi/imag'][0])
    return np.array(out)

def jackknife(samp):                  # delete-1 over config axis -> mean, err_re, err_im
    H = samp.shape[0]
    jk = (samp.sum(0)-samp)/(H-1)
    mean = samp.mean(0)
    ere = np.sqrt(np.maximum((H-1)*np.mean((jk.real-mean.real)**2, 0), 0.0))
    eim = np.sqrt(np.maximum((H-1)*np.mean((jk.imag-mean.imag)**2, 0), 0.0))
    return mean, ere, eim

# sigma_PS per (Nf, mass): contact-subtracted (+2), config-jackknifed.  SIG[nf] = (raw_m, raw_e, sub_m, sub_e).
SIG = {}
for nf in NFS:
    rm = []
    re = []
    sm = []
    se = []
    for md_ in MASS_DIR:
        df = disc_files(nf, md_)
        if len(df) < 2:
            rm.append(np.nan)
            re.append(np.nan)
            sm.append(np.nan)
            se.append(np.nan)
            continue
        etx = cond_cfg(df)
        dens = etx/norm
        sig = dens + np.conj(dens)
        m0, e0, _ = jackknife(sig)
        m2, e2, _ = jackknife(sig + 2.0)
        rm.append(m0.real)
        re.append(e0)
        sm.append(m2.real)
        se.append(e2)
    SIG[nf] = (np.array(rm), np.array(re), np.array(sm), np.array(se))
    print(f'Nf={nf}: light L4 at m_F=0.2 -> {len(disc_files(nf, MASS_DIR[0]))} cfg (NaN = not yet measured)')

Nf=2: light L4 at m_F=0.2 -> 0 cfg (NaN = not yet measured)
Nf=4: light L4 at m_F=0.2 -> 0 cfg (NaN = not yet measured)
Nf=6: light L4 at m_F=0.2 -> 0 cfg (NaN = not yet measured)


In [2]:
# Heavy sea masses (Nf2 only): sigma_PS at effective m_F.  Same disc-byproduct condensate,
# contact-sub +2 (mass-independent; condensate_contact_massive_claude.md Sec 10).
# Defensive: skip masses not yet measured (empty / <2 cfg glob).  x-axis = effective m_F (L4).
HEAVY_DIR = ['0.422900', '0.845799', '1.268699']      # dir mRe string (valence = sea = physical m)
HEAVY_MF = [0.1095, 0.2191, 0.3286]                   # effective m_F at L4 (heavy_mass_L124_impl_plan)
hmf = []
hrm = []
hre = []
hsm = []
hse = []
for md_, mf in zip(HEAVY_DIR, HEAVY_MF):
    df = sorted(glob.glob(esn(2, md_) + 'corr_condensate_eo_nhits1/corr.*.h0.h5'))  # NEW dedicated condensate driver (was corr_ylm_disc_tb2)
    if len(df) < 2:
        print(f'heavy m_F={mf}: {len(df)} cfg -- skipped (not yet measured)')
        continue
    etx = cond_cfg(df)
    dens = etx/norm
    sig = dens + np.conj(dens)
    m0, e0, _ = jackknife(sig)
    m2, e2, _ = jackknife(sig + 2.0)
    hmf.append(mf)
    hrm.append(m0.real)
    hre.append(e0)
    hsm.append(m2.real)
    hse.append(e2)
    print(f'heavy m_F={mf}: {len(df)} cfg  sigma_PS(sub +2) = {m2.real:+.5f} +/- {e2:.1e}')
HEAVY = (np.array(hmf), np.array(hrm), np.array(hre), np.array(hsm), np.array(hse))

heavy m_F=0.1095: 16 cfg  sigma_PS(sub +2) = +0.08513 +/- 1.6e-04
heavy m_F=0.2191: 18 cfg  sigma_PS(sub +2) = +0.16684 +/- 1.2e-04
heavy m_F=0.3286: 21 cfg  sigma_PS(sub +2) = +0.24423 +/- 9.1e-05


In [3]:
# Heavy sigma_FS (Nf2): furnished condensate <sigma_FS> = etadag_xi - xidag_1mDdag_eta
# (condensate_contact_massive_claude.md Eq 303), contact-sub +2 - m_F (FS contact = (m_F-2) V_st, Eq 206).
# Reads corr_condensate_eo_nhits1/ (dedicated e/o condensate driver).  Config-jackknife; x = effective m_F.
def cond_fs_cfg(files):
    out = []
    for fn in files:
        with h5py.File(fn, 'r') as f:
            xi  = f['h0/condensate/etadag_xi/real'][0]        + 1j*f['h0/condensate/etadag_xi/imag'][0]
            xid = f['h0/condensate/xidag_1mDdag_eta/real'][0] + 1j*f['h0/condensate/xidag_1mDdag_eta/imag'][0]
            out.append(xi - xid)
    return np.array(out)

hmf_fs = []
hfs_rm = []
hfs_re = []
hfs_sm = []
hfs_se = []
for md_, mf in zip(HEAVY_DIR, HEAVY_MF):
    df = sorted(glob.glob(esn(2, md_) + 'corr_condensate_eo_nhits1/corr.*.h0.h5'))
    if len(df) < 2:
        print(f'heavy FS m_F={mf}: {len(df)} cfg -- skipped')
        continue
    sig = cond_fs_cfg(df)/norm
    m0, e0, _ = jackknife(sig)
    m2, e2, _ = jackknife(sig + (2.0 - mf))
    hmf_fs.append(mf)
    hfs_rm.append(m0.real)
    hfs_re.append(e0)
    hfs_sm.append(m2.real)
    hfs_se.append(e2)
    print(f'heavy FS m_F={mf}: {len(df)} cfg  sigma_FS(sub +2-mF) = {m2.real:+.5f} +/- {e2:.1e}')
HEAVY_FS = (np.array(hmf_fs), np.array(hfs_rm), np.array(hfs_re), np.array(hfs_sm), np.array(hfs_se))

heavy FS m_F=0.1095: 16 cfg  sigma_FS(sub +2-mF) = -0.00372 +/- 2.1e-05
heavy FS m_F=0.2191: 18 cfg  sigma_FS(sub +2-mF) = -0.01658 +/- 2.6e-05
heavy FS m_F=0.3286: 21 cfg  sigma_FS(sub +2-mF) = -0.03775 +/- 2.6e-05


In [4]:
# Table: sigma_PS contact-subtracted (+2); light (finite only) + heavy Nf2.
print(f"{'m_F':>7} {'Nf':>3} {'contact-sub (+2)':>22}")
for j, m in enumerate(MASS):
    for nf in NFS:
        rm, re, sm, se = SIG[nf]
        if np.isfinite(sm[j]):
            print(f"{m:>7} {nf:>3}   {sm[j]:>+10.5f} +/- {se[j]:.1e}")
    print()
hmf, hrm, hre, hsm, hse = HEAVY
for k in range(len(hmf)):
    print(f"{hmf[k]:>7} {'2h':>3}   {hsm[k]:>+10.5f} +/- {hse[k]:.1e}")

    m_F  Nf       contact-sub (+2)




 0.1095  2h     +0.08513 +/- 1.6e-04
 0.2191  2h     +0.16684 +/- 1.2e-04
 0.3286  2h     +0.24423 +/- 9.1e-05


In [ ]:
# Contact-subtracted sigma_PS vs m_F (L4), LOG-LOG: light Family-B (Nf=2,4,6; finite only) + heavy Nf2.
# (L4 light not yet measured -> only heavy points show.)
fig, ax = plt.subplots()
for i, nf in enumerate(NFS):
    rm, re, sm, se = SIG[nf]
    ok = np.isfinite(sm)
    if ok.sum() >= 1:
        ax.errorbar(MASS[ok], sm[ok], yerr=se[ok], marker=NFMARK[i], ls='none', capsize=3, label=f'Nf={nf}')
    if ok.sum() >= 2:
        slope, icpt = np.polyfit(MASS[ok], sm[ok], 1)
        mg = np.logspace(np.log10(MASS[ok].min()*0.5), np.log10(MASS[ok].max()*1.1), 200)
        ax.plot(mg, slope*mg + icpt, ls='--', lw=0.8, color=ax.lines[-1].get_color())
hmf, hrm, hre, hsm, hse = HEAVY
if len(hmf):
    ax.errorbar(hmf, hsm, yerr=hse, marker='D', color='red', ls='none', capsize=3, label='Nf=2 heavy')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$m_F$')
ax.set_ylabel(r'$\sigma_{PS}$ (contact-subtracted, $+2$)')
ax.set_title(r'$\sigma_{PS}$ contact-subtracted vs $m_F$ (gsq=8, L4, log-log): Nf=2,4,6 + heavy Nf2')
ax.legend()
plt.tight_layout()

In [ ]:
# Contact-subtracted sigma_FS vs m_F, LOG-LOG (heavy Nf2, red diamond).  sigma_FS is uniformly
# NEGATIVE here, so we plot -sigma_FS on log-y.  Order param = sigma_FS/V_st + (2 - m_F).
fig, ax = plt.subplots()
hmf_fs, hfs_rm, hfs_re, hfs_sm, hfs_se = HEAVY_FS
if len(hmf_fs):
    ax.errorbar(hmf_fs, -hfs_sm, yerr=hfs_se, marker='D', color='red', ls='none', capsize=3, label='Nf=2 heavy')
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel(r'$m_F$')
ax.set_ylabel(r'$-\sigma_{FS}$ (contact-subtracted, $+2-m_F$)')
ax.set_title(r'$-\sigma_{FS}$ contact-subtracted vs $m_F$ (gsq=8, L4, log-log): heavy Nf2')
ax.legend()
plt.tight_layout()